In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import curve_fit
import matplotlib as mpl
import matplotlib.pyplot as plt


# Load Data 

In [ ]:
Genta=pd.read_csv('Tables/Gentamicin.csv')
Chp=pd.read_csv('Tables/Chloramphenicol.csv')
Tetra=pd.read_csv('Tables/Tetracycline.csv')
Cipro=pd.read_csv('Tables/Ciprofloxacin.csv')

In [ ]:
qzero_threshold=0.6
Genta.drop(Genta[Genta['prob_neg_drop_zero_det']<qzero_threshold].index,inplace=True)
Chp.drop(Chp[Chp['prob_neg_drop_zero_det']<qzero_threshold].index,inplace=True)
Tetra.drop(Tetra[Tetra['prob_neg_drop_zero_det']<qzero_threshold].index,inplace=True)
Cipro.drop(Cipro[Cipro['prob_neg_drop_zero_det']<qzero_threshold].index,inplace=True)

In [ ]:
Genta_f=pd.read_csv('Tables/Gentamicin_Fit.csv',index_col=0)
Chp_f=pd.read_csv('Tables/Chloramphenicol_Fit.csv',index_col=0)
#Tetra_f=pd.read_csv('Tables/Tetracycline_Fit.csv',index_col=0) # this contains only the seperate fits
Cipro_f=pd.read_csv('Tables/Ciprofloxacin_Fit.csv',index_col=0)

# Fit Full Tetracycline data

In [ ]:
y=Tetra['q_chip_like'].values
c=Tetra['concentration'].values
popt, pcov = curve_fit(lambda t, qz,a,b: 1- (1-qz)* np.exp(-np.power(t/a,b)), c,y,bounds=([0,0,0],[1,np.inf,np.inf]),p0=[0.3,66,3])

In [ ]:
popt, pcov

In [ ]:
xmax=100
x_fitted = np.linspace(0, 100, 5000)
color=sns.color_palette("colorblind")

z=1


fig = plt.figure(figsize=(20, 10), dpi=200)


plt.errorbar(c,y,z* np.sqrt(Tetra['var_q_chip_like'].values),linestyle='')

y_fitted=1-(1-popt[0]) * np.exp(-np.power(x_fitted/popt[1],popt[2]))
plt.plot(x_fitted, y_fitted,linewidth=1,color='gray')

plt.xlim([-1,xmax])

In [ ]:
Tetra_f=pd.DataFrame(np.insert(pcov, 0,popt, axis=0))
Tetra_f=Tetra_f.rename(columns={0:'q_0',1:'a',2:'b'})

# Calculate biological variability

In [ ]:
def biological_variability_weibull(df, params,
                                  date_col="date",
                                  x_col="concentration",
                                  y_col="q_chip_like"):

    q0, a, b = params

    def f(x):
        return 1 - (1 - q0) * np.exp(-np.power(x / a, b))

    df = df.copy()

    # residuals vs your fitted curve
    df["residual"] = df[y_col] - f(df[x_col].values)

    # average per experiment (date)
    date_means = df.groupby(date_col)["residual"].mean()

    bio_sd = date_means.std(ddof=1)

    return bio_sd

In [ ]:
bio_genta = biological_variability_weibull(
    Genta,
    params=(
        Genta_f["q_0"].values[0],
        Genta_f["a"].values[0],
        Genta_f["b"].values[0]
    )
)

bio_Chp = biological_variability_weibull(
    Chp,
    params=(
        Chp_f["q_0"].values[0],
        Chp_f["a"].values[0],
        Chp_f["b"].values[0]
    )
)

bio_Cipro = biological_variability_weibull(
    Cipro,
    params=(
        Cipro_f["q_0"].values[0],
        Cipro_f["a"].values[0],
        Cipro_f["b"].values[0]
    )
)
bio_Tetra = biological_variability_weibull(
    Tetra,
    params=(
        Tetra_f["q_0"].values[0],
        Tetra_f["a"].values[0],
        Tetra_f["b"].values[0]
    )
)

In [ ]:
bio_genta,bio_Chp,bio_Cipro,bio_Tetra